# 📊 Numerical Computing with NumPy: Vektorisierte Anomalie-Detektion für AML

**Lernziel:** Sie wenden NumPy für performante, vektorisierte Berechnungen auf Transaktionsdaten an. Im Fokus stehen:
- Erzeugung und Manipulation von NumPy-Arrays
- Vektorisierte Operationen (keine langsamen Python-Schleifen)
- Statistische Ausreißererkennung (Z-Score, IQR)
- Vektorisierte Datenbereinigung (Log-Transformation, Winsorisierung)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Synthetische Transaktionsbeträge (schiefe Verteilung)
np.random.seed(42)
amounts = np.random.gamma(2, 500, 10000)  # Gamma-verteilte Beträge
amounts = np.round(amounts, 2)

# Manuelle Einfügung von Ausreißern
outlier_indices = [100, 5000, 7500]
amounts[outlier_indices] = [50000, 75000, 120000]

print(f"Anzahl Transaktionen: {len(amounts)}")
print(f"Min: {amounts.min():.2f}, Max: {amounts.max():.2f}")
print(f"Mittelwert: {amounts.mean():.2f}")
print(f"Standardabweichung: {amounts.std():.2f}")

## 1. Vektorisierte Ausreißererkennung mittels Z-Score

Der Z-Score misst, wie viele Standardabweichungen ein Wert vom Mittelwert entfernt ist:
$$ z = \frac{x - \mu}{\sigma} $$

Werte mit $|z| > 3$ gelten oft als Ausreißer.

In [ ]:
# Vektorisierte Berechnung (keine Schleife!)
mean = amounts.mean()
std = amounts.std()
z_scores = (amounts - mean) / std
outliers_z = amounts[np.abs(z_scores) > 3]

print(f"Anzahl Ausreißer (Z-Score > 3): {len(outliers_z)}")
print(f"Ausreißer-Beträge: {outliers_z}")

## 2. Vektorisierte Ausreißererkennung mittels IQR (Interquartilsabstand)

Robust gegenüber extremen Werten. Ausreißer sind Werte außerhalb von $[Q1 - 1.5 \cdot IQR, Q3 + 1.5 \cdot IQR]$.

In [ ]:
q1 = np.percentile(amounts, 25)
q3 = np.percentile(amounts, 75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers_iqr = amounts[(amounts < lower_bound) | (amounts > upper_bound)]

print(f"Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
print(f"Untere Grenze: {lower_bound:.2f}, Obere Grenze: {upper_bound:.2f}")
print(f"Anzahl Ausreißer (IQR): {len(outliers_iqr)}")

## 3. Vektorisierte Datenbereinigung

- **Log-Transformation** zur Reduktion der Schiefe (für viele ML-Modelle vorteilhaft)
- **Winsorisierung** (Begrenzung von Ausreißern auf ein Perzentil)

In [ ]:
# Log-Transformation (nur für positive Werte)
amounts_log = np.log1p(amounts)  # log(1+x) vermeidet log(0)

# Winsorisierung: Begrenze auf 99. Perzentil
p99 = np.percentile(amounts, 99)
amounts_winsorized = np.where(amounts > p99, p99, amounts)

print(f"Vor Winsorisierung: Max = {amounts.max():.2f}")
print(f"Nach Winsorisierung: Max = {amounts_winsorized.max():.2f}")

## 4. Visualisierung der Ausreißer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot der Originaldaten
axes[0].boxplot(amounts, vert=False)
axes[0].set_title('Boxplot der Transaktionsbeträge (mit Ausreißern)')
axes[0].set_xlabel('Betrag in \u20ac')

# Histogramm der log-transformierten Daten
axes[1].hist(amounts_log, bins=50, alpha=0.7)
axes[1].set_title('Log-transformierte Beträge (näherungsweise normalverteilt)')
axes[1].set_xlabel('log(1+Betrag)')

plt.tight_layout()
plt.show()

## Diskussion

- **Z-Score** reagiert empfindlich auf extreme Ausreißer (verzerrt Mittelwert und Std).
- **IQR-Methode** ist robuster und für AML-Daten oft besser geeignet.
- Die **Log-Transformation** macht die Daten symmetrischer – wichtig für viele statistische Verfahren.
- In einer echten AML-Pipeline würden Sie diese vektorisierten Operationen auf Millionen von Transaktionen anwenden – NumPy ist dafür um Größenordnungen schneller als Python-Schleifen.